# Vigil: Track 1 Learning Benchmark

## Google DeepMind × Kaggle "Measuring AGI" Hackathon

This notebook implements the **Track 1: Learning** cognitive benchmark using a stateful graph environment approach.

### Cognitive Abilities Tested:
- **Concept Formation**: Abstracting key features to form categories
- **Associative Learning**: Learning relationships between co-occurring events
- **Reinforcement Learning**: Learning from rewards and punishments

### Key Innovation:
Instead of static QA benchmarks, Vigil creates **stateful cognitive graph environments** where models must explore, learn, and demonstrate cognitive abilities through their actions.

### Submission Instructions:
1. Run all cells to test the benchmark
2. The final cell uses `%choose` to designate the main task for leaderboard submission
3. Save Version to submit to the Kaggle leaderboard

In [ ]:
# Import Kaggle Benchmarks SDK
import kaggle_benchmarks as kbench

# Import Vigil framework
import sys
sys.path.insert(0, '/kaggle/working/vigil')

from vigil.environments.concept_formation import ConceptFormationEnv
from vigil.environments.associative import AssociativeLearningEnv
from vigil.environments.reinforcement import ReinforcementLearningEnv
from vigil.scenarios.loader import ScenarioLoader
from vigil.actions.parser import parse_action
from vigil.actions.schemas import ActionType
from vigil.scoring.profile import CognitiveProfile

print("✓ Vigil benchmark framework loaded successfully")
print(f"  kaggle_benchmarks version: {kbench.__version__ if hasattr(kbench, '__version__') else 'unknown'}")

In [ ]:
# Load and display available scenarios
loader = ScenarioLoader()
scenario_ids = loader.get_scenario_ids()
print(f"Available scenarios: {scenario_ids}")

# Load concept formation config
concept_config = loader.load("concept_formation")
print(f"\nConcept Formation Config:")
print(f"  Scenario ID: {concept_config.get('scenario_id')}")
print(f"  Cognitive Track: {concept_config.get('cognitive_track')}")
print(f"  Sub-ability: {concept_config.get('sub_ability')}")

In [ ]:
# Quick test: Environment instantiation
env = ConceptFormationEnv(scenario_config=concept_config, difficulty=1, seed=42)
state = env.reset()

print("✓ Environment initialized!")
print(f"  Graph nodes: {len(env.graph.nodes)}")
print(f"  Initial budget: {state.budget_remaining}")
print(f"\nAction menu preview:")
print(env.get_available_actions(state)[:200] + "...")

In [ ]:
# Define the main benchmark task for Kaggle
@kbench.task(name="concept_formation_learning")
def concept_formation_task(
    llm,
    difficulty: int = 2,
    seed: int = 42
) -> float:
    """
    Track 1 Learning: Concept Formation Test

    Model must explore graph and infer latent categorization rule.
    Scored 0.0 - 1.0 on correctness, efficiency, evidence, calibration.

    Args:
        llm: LLM to evaluate
        difficulty: Difficulty level (1-3)
        seed: Random seed for reproducibility

    Returns:
        Final score (0.0 - 1.0)
    """
    # Load scenario configuration
    loader = ScenarioLoader()
    config = loader.load("concept_formation")
    
    # Initialize environment
    env = ConceptFormationEnv(
        scenario_config=config,
        difficulty=difficulty,
        seed=seed
    )
    state = env.reset()

    # Exploration loop
    max_turns = 15
    for turn in range(max_turns):
        if state.budget_remaining <= 0:
            break
        
        # Present current state
        prompt = f"""
You are exploring an unknown graph to discover a hidden pattern.
The graph contains nodes belonging to hidden categories.
Nodes in the same category share core features.
Your task is to infer the categorization rule.

{env.get_available_actions(state)}

Your action (use format 'expand:node_id' or 'submit' when ready):
"""
        response = llm.prompt(prompt)
        
        # Parse and execute action
        action = parse_action(response)
        
        if action and action.is_valid():
            success, obs = env.execute_action(state, action)
            
            if action.action_type == ActionType.SUBMIT:
                # Get final hypothesis
                final_prompt = """
Based on your exploration, what is the hidden rule for how nodes are categorized?
Describe the pattern you discovered.
"""
                final_answer = llm.prompt(final_prompt)
                scores = env.score_exploration(state, final_answer)
                return scores["final_score"]

    # Timeout - model didn't submit
    return 0.0


print("✓ Task 'concept_formation_learning' defined")

In [ ]:
# Test run with default LLM (optional - comment out for submission)
print("Running test evaluation (difficulty=1 for quick test)...")
try:
    result = concept_formation_task.run(kbench.llm, difficulty=1, seed=42)
    print(f"✓ Test result: {result:.3f}")
except Exception as e:
    print(f"Note: Test skipped or failed (expected in some environments): {e}")

In [ ]:
# Dataset evaluation pattern (for full benchmark runs)
import pandas as pd

@kbench.task(name="single_graph_instance")
def single_instance(llm, difficulty: int, seed: int) -> bool:
    """Evaluates model on one graph instance."""
    score = concept_formation_task.run(llm, difficulty=difficulty, seed=seed)
    return score > 0.5  # Pass threshold


@kbench.task(name="concept_formation_full_benchmark")
def full_benchmark(llm, df: pd.DataFrame) -> tuple[float, float]:
    """
    Runs all graph instances and returns (mean_score, std).
    Uses Kaggle's .evaluate() for parallel execution.
    """
    with kbench.client.enable_cache():
        runs = single_instance.evaluate(
            llm=[llm],
            evaluation_data=df,
            n_jobs=4,
            timeout=300,
            remove_run_files=True
        )

    eval_df = runs.as_dataframe()
    mean_score = float(eval_df.result.mean())
    std_score = float(eval_df.result.std())

    return mean_score, std_score


def create_concept_dataset(n_instances: int = 10) -> pd.DataFrame:
    """Create dataset of graph configurations."""
    configs = []
    for i in range(n_instances):
        configs.append({
            "difficulty": (i % 3) + 1,  # Cycle through difficulties
            "seed": i + 42
        })
    return pd.DataFrame(configs)


print("✓ Dataset evaluation tasks defined")

In [ ]:
# Additional tasks for other cognitive abilities

@kbench.task(name="associative_learning")
def associative_learning_task(
    llm,
    difficulty: int = 2,
    seed: int = 42
) -> float:
    """
    Track 1 Learning: Associative Learning Test
    
    Model must learn relationships between co-occurring events.
    """
    loader = ScenarioLoader()
    config = loader.load("associative")
    
    env = AssociativeLearningEnv(
        scenario_config=config,
        difficulty=difficulty,
        seed=seed
    )
    state = env.reset()
    
    max_turns = 12
    for turn in range(max_turns):
        if state.budget_remaining <= 0:
            break
        
        prompt = f"""
You are exploring to discover associations between nodes.
Certain nodes appear together in pairs - your task is to find the pattern.

{env.get_available_actions(state)}

Your action:
"""
        response = llm.prompt(prompt)
        action = parse_action(response)
        
        if action and action.is_valid():
            success, obs = env.execute_action(state, action)
            
            if action.action_type == ActionType.SUBMIT:
                final_answer = llm.prompt("Describe the association pattern:")
                scores = env.score_exploration(state, final_answer)
                return scores["final_score"]
    
    return 0.0


@kbench.task(name="reinforcement_learning")
def reinforcement_learning_task(
    llm,
    difficulty: int = 2,
    seed: int = 42
) -> float:
    """
    Track 1 Learning: Reinforcement Learning Test
    
    Model must learn from rewards and punishments.
    """
    loader = ScenarioLoader()
    config = loader.load("reinforcement")
    
    env = ReinforcementLearningEnv(
        scenario_config=config,
        difficulty=difficulty,
        seed=seed
    )
    state = env.reset()
    
    max_turns = 15
    for turn in range(max_turns):
        if state.budget_remaining <= 0:
            break
        
        prompt = f"""
You are navigating an environment with rewards and penalties.
Find the reward nodes and avoid penalty nodes.

{env.get_available_actions(state)}

Your action:
"""
        response = llm.prompt(prompt)
        action = parse_action(response)
        
        if action and action.is_valid():
            success, obs = env.execute_action(state, action)
            
            if action.action_type == ActionType.SUBMIT:
                final_answer = llm.prompt("Describe the reward structure:")
                scores = env.score_exploration(state, final_answer)
                return scores["final_score"]
    
    return 0.0


print("✓ Additional learning tasks defined")

## Leaderboard Submission

The `%choose` magic command designates which task should be submitted to the Kaggle leaderboard.

**Important:** This must be the last cell in the notebook for proper submission.

In [ ]:
# Designate main task for leaderboard submission
%choose concept_formation_learning